In [13]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

#Step 1: Load the Dataset

df = pd.read_csv("Dataset-Table 1.csv")

#Step 2: Inspect the Dataset

print("First 5 Rows:")
print(df.head())

print("\nDataset Information:")
print(df.info())

print("\nStatistical Summary:")
print(df.describe())

#Step 3: Check the number of rows and columns

print("\nRows and Columns:")
print(df.shape)

print("Number of Rows:", df.shape[0])
print("Number of Columns:", df.shape[1])

#Step 4: Check data types 

print("\nData Types:")
print(df.dtypes)

# Remove duplicates
df = df.drop_duplicates()

# Drop empty column
df = df.drop(columns=["Unnamed: 21"])

# Drop Customer ID
df = df.drop(columns=["Customer_ID"])

# Convert currency columns
df["Annual_Income"] = (
    df["Annual_Income"]
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(float)
)

df["Average_Order_Value"] = (
    df["Average_Order_Value"]
    .str.replace("$", "", regex=False)
    .astype(float)
)

# Convert date
df["Campaign_Date"] = pd.to_datetime(df["Campaign_Date"])

# Extract month (optional)
df["Campaign_Month"] = df["Campaign_Date"].dt.month

# Remove original date
df = df.drop(columns=["Campaign_Date"])

# Convert Yes/No to 1/0
binary_cols = [
    "Email_Opened",
    "Ad_Clicked",
    "Prior_Campaign_Response",
    "Converted"
]

for col in binary_cols:
    df[col] = df[col].map({"Yes": 1, "No": 0})

# Fill missing values
df["Annual_Income"].fillna(df["Annual_Income"].median(), inplace=True)
df["Average_Order_Value"].fillna(df["Average_Order_Value"].median(), inplace=True)
df["Customer_Satisfaction"].fillna(df["Customer_Satisfaction"].median(), inplace=True)

df["Customer_Segment"].fillna(
    df["Customer_Segment"].mode()[0],
    inplace=True
)

# One-hot encoding
df = pd.get_dummies(
    df,
    columns=[
        "Customer_Segment",
        "Region",
        "Campaign_Channel",
        "Campaign_Type"
    ],
    drop_first=True
)

#Step 9: Handle categorical & numerical variables 

X = df.drop("Converted", axis=1)
y = df["Converted"]

#Step 10: Split the data into training and testing sets 

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42
)

model = DecisionTreeClassifier(
    criterion="gini",
    max_depth=5,
    random_state=42
)

model.fit(X_train, y_train) 

First 5 Rows:
  Customer_ID Campaign_Date  Age Annual_Income Customer_Segment  \
0   CUST-0001    2026-04-14   36       $58,196              New   
1   CUST-0002    2026-04-24   38       $49,043        Returning   
2   CUST-0003    2026-02-01   21       $45,952              New   
3   CUST-0004    2026-02-28   18      $107,638            Loyal   
4   CUST-0005    2026-03-01   50       $89,969        Returning   

             Region Campaign_Channel    Campaign_Type  Ad_Exposure_Count  \
0  British Columbia            Email  Brand Awareness                  4   
1           Ontario            Email      Retargeting                  5   
2           Ontario            Email   Discount Offer                  4   
3  British Columbia       Search Ads      Retargeting                  4   
4            Quebec       Search Ads      New Product                  1   

  Email_Opened  ... Previous_Purchases  Average_Order_Value  \
0          Yes  ...                  1              $131.51   


DecisionTreeClassifier(max_depth=5, random_state=42)

In [14]:
y_pred = model.predict(X_test)

print("Confusion Matrix")
print(confusion_matrix(y_test, y_pred))

print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))

print("\nClassification Report")
print(classification_report(y_test, y_pred))



Confusion Matrix
[[65 29]
 [43 43]]

Accuracy: 0.6
Precision: 0.5972222222222222
Recall: 0.5
F1 Score: 0.5443037974683543

Classification Report
              precision    recall  f1-score   support

           0       0.60      0.69      0.64        94
           1       0.60      0.50      0.54        86

    accuracy                           0.60       180
   macro avg       0.60      0.60      0.59       180
weighted avg       0.60      0.60      0.60       180



In [15]:
feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_
})

print(feature_importance.sort_values(
    by="Importance",
    ascending=False
))

                          Feature  Importance
10                  Loyalty_Score    0.232179
12  Social_Media_Engagement_Score    0.211247
1                   Annual_Income    0.122643
9        Days_Since_Last_Purchase    0.116695
4                      Ad_Clicked    0.080673
5     Website_Visits_Last_30_Days    0.074799
11          Customer_Satisfaction    0.036500
6              Previous_Purchases    0.032745
2               Ad_Exposure_Count    0.025354
14                 Campaign_Month    0.023737
23           Campaign_Channel_SMS    0.018612
21                  Region_Quebec    0.015510
0                             Age    0.009306
13        Prior_Campaign_Response    0.000000
8        Discount_Offered_Percent    0.000000
15         Customer_Segment_Loyal    0.000000
16           Customer_Segment_New    0.000000
17     Customer_Segment_Returning    0.000000
18        Region_British Columbia    0.000000
19                 Region_Ontario    0.000000
20                   Region_Other 